# Neighbor harm reward

Does explicitly penalizing the pollution an agent sends to its neighbors change behavior? The earlier local-versus-global comparison mixed several changes in the learning signal. This notebook keeps the agent's own exposure and mitigation-cost terms fixed and changes only the weight on neighbor harm. The experiment remains synthetic and exploratory.

For region `i`, let `e_i` be its emissions, `s` the spillover coefficient, `x'_j` a neighbor's new load, and `E(x) = x + 0.2 max(0, x - 30)^2`. We use:

```text
neighbor_harm_i = sum(E(x'_j) - E(x'_j - s * e_i) for j in neighbors(i))
reward_i(weight) = -E(x'_i) - 2 * cost_i - weight * neighbor_harm_i
```

The counterfactual removes only region `i`'s contribution to each neighbor's *current* load. This is a one-step reward signal, not a full counterfactual future trajectory. At spillover 0, `neighbor_harm_i = 0` for every action and state; all weights then give exactly the local reward. At spillover 0.2, weight 0 is local reward, while weights 0.5 and 1.0 add half or all of this one-step neighbor penalty. These weights were chosen before examining the new runs.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from study_1_emissions_coordination.config import EnvironmentConfig, TrainingConfig
from study_1_emissions_coordination.evaluate import evaluate_policy, summarize_by_policy, summarize_by_training_seed, write_csv
from study_1_emissions_coordination.train import save_agents, train_iql

In [ ]:
train_new_runs = False
spillover = 0.2
new_weights = {"externality_05": 0.5, "externality_10": 1.0}
training_seeds = (11, 23, 37, 41, 53, 67, 71, 83, 97, 101, 107,
                  109, 127, 131, 137, 149, 157, 163, 173, 179)
evaluation_seeds = tuple(range(100000, 100020))
saved_comparison_directory = "study_1_emissions_coordination/results/followup/spillover_02"
saved_zero_directory = "study_1_emissions_coordination/results/followup/spillover_00"
output_directory = "study_1_emissions_coordination/results/notebook_runs/neighbor_externality_01"

environment_config = EnvironmentConfig()
environment_config.spillover = spillover
training_config = TrainingConfig()
training_config.training_seeds = training_seeds
training_config.evaluation_seeds = evaluation_seeds

with open(f"{saved_comparison_directory}/manifest.json", encoding="utf-8") as file:
    original_manifest = json.load(file)
if original_manifest["environment"]["spillover"] != spillover:
    raise ValueError("The saved comparison uses a different spillover.")
if tuple(original_manifest["training"]["training_seeds"]) != training_seeds:
    raise ValueError("The saved comparison uses different training seeds.")
if tuple(original_manifest["training"]["evaluation_seeds"]) != evaluation_seeds:
    raise ValueError("The saved comparison uses different evaluation seeds.")
if original_manifest["training"]["training_episodes"] != training_config.training_episodes:
    raise ValueError("The saved comparison uses a different episode count.")

print("Train new runs:", train_new_runs)
print("New neighbor-harm weights:", new_weights)
print("Training seeds:", len(training_seeds), "Held-out demand seeds:", len(evaluation_seeds))

## Train and evaluate the new reward policies

Reuse the 20 saved local-reward policies as weight 0. Train separate agents for weights 0.5 and 1.0 at spillover 0.2. All conditions use the same training seeds, 2,000 episodes, 50 steps per episode, observation space, action costs, and held-out demand seeds. Saved Q-tables make each new learned policy inspectable. The output folder is prepared separately.

In [ ]:
if train_new_runs:
    manifest = {
        "question": "Neighbor harm reward at spillover 0.2",
        "source": "study_1_emissions_coordination/notebooks/05_neighbor_harm_reward.ipynb",
        "command": "Run all cells from the repository root",
        "environment": vars(environment_config),
        "training": vars(training_config),
        "new_weights": new_weights,
        "reused_weight_zero": saved_comparison_directory,
        "zero_spillover_reference": saved_zero_directory,
        "models": [f"q_{name}_{seed}.json" for name in new_weights
                   for seed in training_seeds],
        "reward_formula": "local_reward_i - weight * sum_neighbor_counterfactual_exposure_increase",
        "evaluation": "20 common held-out demand seeds per training seed",
        "status": "Exploratory follow-up",
    }
    with open(f"{output_directory}/manifest.json", "x", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
        file.write("\n")

    episode_rows = []
    for name, weight in new_weights.items():
        for seed in training_seeds:
            agents = train_iql("externality", seed, training_config,
                               environment_config, weight)
            save_agents(f"{output_directory}/q_{name}_{seed}.json", agents)

            def greedy_policy(observations, agents=agents):
                return tuple(agent.select_action(observation, greedy=True)
                             for agent, observation in zip(agents, observations))

            for metrics in evaluate_policy(greedy_policy, evaluation_seeds,
                                           environment_config):
                episode_rows.append({"policy": name, "training_seed": seed, **metrics})
            print(f"Trained {name}, seed {seed}", flush=True)

    run_rows = summarize_by_training_seed(episode_rows)
    policy_rows = summarize_by_policy(run_rows)
    write_csv(f"{output_directory}/episodes.csv", episode_rows)
    write_csv(f"{output_directory}/run_summary.csv", run_rows)
    write_csv(f"{output_directory}/policy_summary.csv", policy_rows)
    print(f"Saved {len(episode_rows)} held-out episodes.")
else:
    print("Reading the saved neighbor-harm results.")

## Compare outcomes at spillover 0.2

The new models are evaluated using common outcomes, not their differently defined training rewards. The original local, mixed, and global policies are shown as references; this experiment's controlled comparison is **neighbor-harm weight 0 versus 0.5 versus 1.0**.

In [ ]:
original_summary = pd.read_csv(f"{saved_comparison_directory}/policy_summary.csv")
original_summary = original_summary[
    original_summary["policy"].isin(("iql_local", "iql_mixed", "iql_global"))
]
new_summary = pd.read_csv(f"{output_directory}/policy_summary.csv")
comparison = pd.concat([original_summary, new_summary], ignore_index=True)
display(comparison[
    ["policy", "runs", "mean_regional_pollution_mean", "mean_regional_pollution_sd",
     "mitigation_cost_mean", "mitigation_cost_sd", "threshold_exceedance_mean",
     "action_0_fraction_mean", "action_2_fraction_mean"]
].round(3))

## Pair each new training run with its local-reward reference

Each difference below uses the same training seed and the same 20 held-out demand seeds. A negative pollution difference means the neighbor-harm reward produced less pollution; a positive cost difference means it spent more. The standard deviation is across 20 paired training runs.

In [ ]:
local_runs = pd.read_csv(f"{saved_comparison_directory}/run_summary.csv")
local_runs = local_runs[local_runs["policy"] == "iql_local"]
new_runs = pd.read_csv(f"{output_directory}/run_summary.csv")
paired = new_runs.merge(local_runs, on="training_seed", suffixes=("_new", "_local"))
paired_rows = []
for _, row in paired.iterrows():
    paired_rows.append({
        "policy": row["policy_new"],
        "training_seed": int(row["training_seed"]),
        "pollution_gap": row["mean_regional_pollution_new"]
                         - row["mean_regional_pollution_local"],
        "cost_gap": row["mitigation_cost_new"] - row["mitigation_cost_local"],
        "threshold_gap": row["threshold_exceedance_new"]
                         - row["threshold_exceedance_local"],
    })
paired_results = pd.DataFrame(paired_rows)
paired_summary = paired_results.groupby("policy").agg(
    pollution_gap_mean=("pollution_gap", "mean"),
    pollution_gap_sd=("pollution_gap", "std"),
    cost_gap_mean=("cost_gap", "mean"),
    cost_gap_sd=("cost_gap", "std"),
    threshold_gap_mean=("threshold_gap", "mean"),
    threshold_gap_sd=("threshold_gap", "std"),
    lower_pollution_runs=("pollution_gap", lambda values: (values < 0).sum()),
    higher_cost_runs=("cost_gap", lambda values: (values > 0).sum()),
    paired_runs=("training_seed", "size"),
)
display(paired_summary.round(3))
print("Paired runs compare the same training seed.")

## Changes relative to local reward

The points show average **paired** differences from the local-reward runs. Vertical bars show one standard deviation across 20 training seeds. The zero-weight point is exactly zero by definition. A lower pollution point usually comes with a higher mitigation cost here.

In [ ]:
weights = (0.0, 0.5, 1.0)
pollution_means = (0.0, paired_summary.loc["externality_05", "pollution_gap_mean"],
                   paired_summary.loc["externality_10", "pollution_gap_mean"])
pollution_sds = (0.0, paired_summary.loc["externality_05", "pollution_gap_sd"],
                 paired_summary.loc["externality_10", "pollution_gap_sd"])
cost_means = (0.0, paired_summary.loc["externality_05", "cost_gap_mean"],
              paired_summary.loc["externality_10", "cost_gap_mean"])
cost_sds = (0.0, paired_summary.loc["externality_05", "cost_gap_sd"],
            paired_summary.loc["externality_10", "cost_gap_sd"])

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].errorbar(weights, pollution_means, yerr=pollution_sds, fmt="o-", capsize=4)
axes[0].axhline(0, color="gray", linewidth=1)
axes[0].set(xlabel="Neighbor-harm weight", ylabel="Pollution difference",
            title="New reward minus local")
axes[1].errorbar(weights, cost_means, yerr=cost_sds, fmt="o-", capsize=4)
axes[1].axhline(0, color="gray", linewidth=1)
axes[1].set(xlabel="Neighbor-harm weight", ylabel="Cost difference",
            title="New reward minus local")
for axis in axes:
    axis.set_xticks(weights)
    axis.grid(alpha=0.2)
figure.tight_layout()
if train_new_runs:
    figure.savefig(f"{output_directory}/paired_effects.svg")
display(figure)
plt.close(figure)

## Pollution and cost together

Bars show one standard deviation across 20 training seeds. They are not confidence intervals. Original local, mixed, and global policies appear as context; the neighbor-harm weights are the controlled variation.

In [ ]:
figure, axis = plt.subplots(figsize=(9, 6))
for _, row in comparison.iterrows():
    axis.errorbar(row["mitigation_cost_mean"], row["mean_regional_pollution_mean"],
                  xerr=row["mitigation_cost_sd"],
                  yerr=row["mean_regional_pollution_sd"],
                  fmt="o", capsize=4, label=row["policy"])
axis.set(xlabel="Mitigation cost per episode",
         ylabel="Mean regional pollution",
         title="Neighbor-harm rewards at spillover 0.2")
axis.grid(alpha=0.2)
axis.legend(fontsize=8)
figure.tight_layout()
if train_new_runs:
    figure.savefig(f"{output_directory}/pollution_vs_cost.svg")
display(figure)
plt.close(figure)

## Zero-spillover comparison

When spillover is zero, removing region `i`'s emissions from a neighbor's new load changes that load by `0 * e_i = 0`. The neighbor-harm term is therefore exactly zero, and the 0, 0.5, and 1.0 reward formulas are identical to local reward. With matching training seeds and random generators, they would produce the same Q-tables and held-out outcomes. This is a mathematical identity of this reward design; we reuse the already saved zero-spillover local policies instead of training duplicate copies. The table below shows those shared outcomes.

In [ ]:
zero_summary = pd.read_csv(f"{saved_zero_directory}/policy_summary.csv")
zero_local = zero_summary[zero_summary["policy"] == "iql_local"].copy()
zero_local["reward_weights_with_identical_formula"] = "0, 0.5, 1.0"
display(zero_local[
    ["reward_weights_with_identical_formula", "runs",
     "mean_regional_pollution_mean", "mitigation_cost_mean",
     "threshold_exceedance_mean"]
].round(3))

## Interpretation

With weight 0.5, mean pollution was 26.25 and mean mitigation cost was 529, nearly the same as the local-reward reference (26.18 and 527). The paired pollution difference was +0.07 with a standard deviation of 0.84 across training seeds; only 8 of 20 runs had lower pollution. At weight 1.0, mean pollution was 25.88 and cost was 578. Relative to the same-seed local policies, that was **0.30 less pollution** and **50 more cost** on average. Pollution fell in 15 of 20 paired runs, while cost rose in 16. High mitigation was chosen in 12.5% of region-steps, compared with 8.6% for local reward.

These results suggest that a full one-step neighbor-harm penalty encouraged more mitigation in this setting, with a pollution-versus-cost tradeoff. The half-weight setting did not show a clear improvement. The paired pollution difference at weight 1.0 also varied substantially across seeds (standard deviation 0.86), so we should not treat the mean as a reliable advantage across other settings. At zero spillover, all tested weights reduce **exactly** to the local reward by construction; this removes the reward-scaling confound of the earlier global-versus-local zero-spillover comparison. The experiment identifies the effect of this particular reward term in the synthetic model, not a real-world externality or an optimal weight.

## Save the tables

When `train_new_runs = True`, the experiment cell saves Q-tables, per-episode values, summaries, and a manifest. This final cell saves the paired comparisons. After the first run, set `train_new_runs = False` to inspect results without changing saved files. To run a new experiment, prepare another output folder and change `output_directory`.

In [ ]:
if train_new_runs:
    paired_results.to_csv(f"{output_directory}/paired_vs_local.csv", index=False)
    paired_summary.to_csv(f"{output_directory}/paired_summary.csv")
    print("Saved paired comparison tables.")
else:
    print("Saved outputs remain unchanged.")

## Conclusion

Explicitly charging each region for the harm its emissions cause to its neighbors changed the learned policies when pollution could spill across borders. At spillover 0.2, a full neighbor-harm weight reduced mean pollution by about 0.30 relative to local reward across 20 paired training seeds, but raised mean mitigation cost by about 50. The half-weight setting showed no clear pollution improvement.

At zero spillover, the neighbor-harm term is exactly zero, so these reward formulas become identical to local reward. This makes the comparison easier to interpret: the change at spillover 0.2 comes from adding a spillover-dependent reward term while keeping the region's own cost term fixed. The results are still variable across seeds and come from a small synthetic environment. Notebook 06 checks other spillover strengths and shows the associated pollution and cost tradeoffs.